# Laboratorio 4 ML — Asistente de compras por voz (ESCI + RAG + agente + TTS clonado)

Asistente que recibe consultas habladas en español (ASR), recupera productos del catálogo ESCI (RAG), ejecuta
operaciones sobre una tienda simulada mediante un agente con herramientas y responde por voz con una voz clonada (TTS).

**Parámetros:** `configuracion.py` (misma carpeta). Edita ese archivo, reinicia el kernel y ejecuta de arriba a abajo.
Lo ya ejecutado se omite (`output/resultados/registro.json`).

**Carpetas:** `input/` (data ESCI, voz de referencia y audios de los escenarios) · `output/` (datos, índice, tienda,
escenarios, resultados) · `src/` (código del grupo) · `docs/` (diseño, guía de grabación y guía del video).

**Secciones:** 0 Enunciado · 1 Configuración y entorno · 2 Datos · 3 Tienda · 4 Índice RAG · 5 Evaluación de la
recuperación (modelo base y ablación) · 6 Modelos de lenguaje y voz · 7 Agente y autoprueba · 8 Escenarios por texto ·
9 Escenarios por voz y repetición con transcripción correcta · 10 Métricas · 11 Gráficos · 12 Análisis de aciertos y
fallos · 13 Demo interactiva · 14 Instrucciones, versiones y diferencias

## 0. Enunciado
*(pegar aquí el enunciado del Laboratorio 4)*

## 1. Configuración, bitácora y entorno

In [ ]:
import importlib, json, platform, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import configuracion as C
importlib.reload(C)
from src.bitacora import Bitacora, hash_cfg, hms

for r in (C.RUTA_DATOS, C.RUTA_EVAL, C.RUTA_INDICE, C.RUTA_TIENDA, C.RUTA_ESC, C.RUTA_RES, C.RUTA_AUDIOS):
    r.mkdir(parents=True, exist_ok=True)
FIG = C.RUTA_RES / "figuras"; FIG.mkdir(exist_ok=True)
BIT = Bitacora(C.RUTA_RES, C.FORZAR_REEJECUTAR)

def guardar_fig(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=150, bbox_inches="tight"); print("Figura:", FIG / f"{nombre}.png")

try:
    import torch
    CUDA = torch.cuda.is_available()
    GPU = f"{torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB)" if CUDA else "sin GPU"
except ImportError:
    torch, CUDA, GPU = None, False, "torch no instalado"
DISP = ("cuda" if CUDA else "cpu") if C.DISPOSITIVO == "auto" else C.DISPOSITIVO

def vram():
    return round(torch.cuda.memory_allocated() / 2**30, 2) if CUDA else 0.0

VERSIONES = {"python": sys.version.split()[0], "sistema": platform.platform(), "gpu": GPU}
for m in ("torch", "transformers", "sentence_transformers", "accelerate", "bitsandbytes", "TTS", "rank_bm25", "jiwer", "pandas", "numpy"):
    try:
        VERSIONES[m] = importlib.import_module(m).__version__
    except Exception as e:
        VERSIONES[m] = f"no disponible ({type(e).__name__})"
(C.RUTA_RES / "versiones.json").write_text(json.dumps(VERSIONES, indent=2), encoding="utf-8")
pd.Series(VERSIONES).to_frame("version")

## 2. Datos: catálogo, consultas y etiquetas
- ESCI **small** filtrado por **español**. Consultas separadas por `query_id`: **eval** = split *test* (≥ 200), **dev** = split *train*.
- Catálogo = todos los productos anotados de esas consultas (+ distractores si faltara para llegar a 5 000).
- Las etiquetas ESCI se guardan aparte en `output/datos/evaluacion/qrels.parquet` y **solo** las usa la evaluación;
  el agente únicamente accede a `catalogo.parquet` y a la tienda.

In [ ]:
from src.datos import preparar
CFG_DATOS = {"n_eval": C.N_CONSULTAS_EVAL, "n_dev": C.N_CONSULTAS_DEV, "min": C.MIN_CATALOGO, "semilla": C.SEMILLA}
H_DATOS = BIT.ejecutar("2. Datos (catalogo, consultas, qrels)", CFG_DATOS,
                       [C.RUTA_DATOS / "catalogo.parquet", C.RUTA_DATOS / "consultas.parquet", C.RUTA_EVAL / "qrels.parquet"],
                       lambda: (C.RUTA_RES / "resumen_datos.json").write_text(json.dumps(
                           preparar(C.RUTA_INPUT, C.RUTA_DATOS, C.RUTA_EVAL, C.N_CONSULTAS_EVAL, C.N_CONSULTAS_DEV,
                                    C.MIN_CATALOGO, C.SEMILLA), indent=1, ensure_ascii=False), encoding="utf-8"))
CAT = pd.read_parquet(C.RUTA_DATOS / "catalogo.parquet")
CONS = pd.read_parquet(C.RUTA_DATOS / "consultas.parquet")
QRELS = pd.read_parquet(C.RUTA_EVAL / "qrels.parquet")
assert "esci_label" not in CAT.columns, "el catalogo no debe contener etiquetas"
print(json.loads((C.RUTA_RES / "resumen_datos.json").read_text(encoding="utf-8")))
CAT.head(3)

## 3. Tienda simulada (semilla fija y restaurable)
Precios (log-normal, terminados en ,99) y existencias (8 % agotados) generados con `SEMILLA`. La base queda en
`tienda_base.sqlite`; `restaurar()` la copia sobre `tienda.sqlite`. La huella (hash de inventario y pedidos) demuestra
que dos restauraciones dan el mismo estado.

In [ ]:
from src.tienda import Tienda
BASE, TRABAJO = C.RUTA_TIENDA / "tienda_base.sqlite", C.RUTA_TIENDA / "tienda.sqlite"
CFG_TIENDA = {"datos": H_DATOS, "semilla": C.SEMILLA, "precio": C.PRECIO_LOGNORMAL, "rango": C.PRECIO_RANGO,
              "agotado": C.PROB_AGOTADO, "stock": C.STOCK_MAX}
BIT.ejecutar("3. Tienda simulada (base con semilla)", CFG_TIENDA, [BASE],
             lambda: Tienda.crear_base(BASE, CAT["product_id"], C.SEMILLA, *C.PRECIO_LOGNORMAL, C.PRECIO_RANGO,
                                       C.PROB_AGOTADO, C.STOCK_MAX))
TIENDA = Tienda(BASE, TRABAJO, C.CANTIDAD_MAX)
with BIT.paso("3.1 Restauracion reproducible") as info:
    TIENDA.restaurar(); h1 = TIENDA.huella()
    TIENDA.fijar_stock(CAT["product_id"][0], 0)
    TIENDA.restaurar(); h2 = TIENDA.huella()
    assert h1 == h2
    info["comentario"] = f"huella {h1}"
inv = pd.read_sql("SELECT precio_cent / 100.0 AS precio, stock FROM productos", TIENDA.con)
print(f"Huella: {h1} | agotados: {(inv['stock'] == 0).mean():.1%}")
inv.describe().round(2).T

## 4. Índice RAG
- **BM25** (modelo base léxico) sobre título, marca, color, características y descripción.
- **Denso:** `MODELO_EMB` (e5, prefijos `query:`/`passage:`); los embeddings del catálogo se guardan en `output/indice/`.
- **Reranker:** `MODELO_RERANK` (cross-encoder multilingüe) sobre los mejores `TOP_RERANK` candidatos híbridos.

In [ ]:
CARGA = []   # carga inicial de cada modelo (separada de las consultas posteriores)
def registrar_carga(nombre, etapa, fn):
    v0, t0 = vram(), time.perf_counter()
    obj = fn()
    CARGA.append({"modelo": nombre, "etapa": etapa, "carga_s": round(time.perf_counter() - t0, 2),
                  "vram_gb": round(vram() - v0, 2), "vram_total_gb": vram()})
    print(CARGA[-1])
    return obj

from src.recuperacion import Recuperador
with BIT.paso("4.1 Indice BM25"):
    REC = registrar_carga("BM25", "recuperacion", lambda: Recuperador(CAT, C.MAX_CHARS_TEXTO))
with BIT.paso("4.2 Modelo denso + embeddings del catalogo", C.MODELO_EMB):
    registrar_carga(C.MODELO_EMB, "recuperacion", lambda: REC.cargar_denso(C.MODELO_EMB, DISP, C.RUTA_INDICE))
with BIT.paso("4.3 Reranker", C.MODELO_RERANK):
    registrar_carga(C.MODELO_RERANK, "recuperacion", lambda: REC.cargar_reranker(C.MODELO_RERANK, DISP))
print("Embeddings:", REC.emb.shape)

## 5. Evaluación de la recuperación: modelo base y ablación
- **Métrica:** nDCG@10 sobre los **candidatos anotados** de cada consulta, ganancias E = 1, S = 0,1, C = 0,01, I = 0.
- **Ajuste solo con dev:** α (peso denso vs BM25) y β (peso reranker). Las consultas eval se usan una sola vez.
- **Comparación:** BM25 (modelo base) vs denso vs híbrido vs **sistema** (híbrido + reranker).
- **Ablación** (mismas consultas): sistema **sin reranker** (= híbrido) y sistema **sin denso** (BM25 + reranker).
- Complemento: nDCG@10 recuperando de **todo el catálogo** (no anotados = 0), que es lo que vive el agente.

In [ ]:
from src.recuperacion import precomputar, evaluar, ajustar, evaluar_catalogo
RUTA_REC = C.RUTA_RES / "recuperacion_eval.csv"
CFG_REC = {"datos": H_DATOS, "emb": C.MODELO_EMB, "rr": C.MODELO_RERANK, "grid": C.GRID, "k": C.K_NDCG,
           "gan": C.GANANCIAS, "top": [C.TOP_CANDIDATOS, C.TOP_RERANK]}

def evaluar_recuperacion():
    dev, ev = CONS[CONS["conjunto"] == "dev"], CONS[CONS["conjunto"] == "eval"]
    pre_dev = precomputar(REC, dev, QRELS, C.GANANCIAS)
    mejor, grid = ajustar(pre_dev, C.K_NDCG, C.GRID)
    grid.to_csv(C.RUTA_RES / "recuperacion_ajuste_dev.csv", index=False)
    (C.RUTA_RES / "recuperacion_parametros.json").write_text(json.dumps(mejor, indent=1), encoding="utf-8")
    t0 = time.perf_counter(); pre_ev = precomputar(REC, ev, QRELS, C.GANANCIAS)
    t_pre = (time.perf_counter() - t0) / len(ev)
    variantes = [("BM25 (modelo base)", "bm25", 0, 0), ("Denso e5", "denso", 0, 0),
                 ("Hibrido = sistema sin reranker (ablacion)", "hibrido", mejor["alpha"], 0),
                 ("BM25 + reranker = sistema sin denso (ablacion)", "bm25+reranker", 0, mejor["beta_bm25+reranker"]),
                 ("Sistema: hibrido + reranker", "sistema", mejor["alpha"], mejor["beta_sistema"])]
    filas, por_q = [], []
    for nombre, m, a, b in variantes:
        r_dev = evaluar(pre_dev, m, a, b, C.K_NDCG)["ndcg"].mean()
        r = evaluar(pre_ev, m, a, b, C.K_NDCG)
        por_q.append(r.assign(metodo=m))
        REC.alpha = a
        cat_ev = evaluar_catalogo(REC, ev, QRELS, m, C.GANANCIAS, C.K_NDCG, beta=b)["ndcg"].mean() if m != "denso" or REC.encoder else np.nan
        filas.append({"variante": nombre, "metodo": m, "alpha": a, "beta": b, "nDCG@10 dev": r_dev,
                      "nDCG@10 eval (candidatos anotados)": r["ndcg"].mean(), "nDCG@10 eval (catalogo completo)": cat_ev})
    pd.DataFrame(filas).to_csv(RUTA_REC, index=False)
    pd.concat(por_q).to_csv(C.RUTA_RES / "recuperacion_por_consulta.csv", index=False)

if C.EJECUTAR_EVAL_RECUPERACION:
    BIT.ejecutar("5. Evaluacion de recuperacion (ajuste dev, eval una vez)", CFG_REC, [RUTA_REC], evaluar_recuperacion)
PARAM = json.loads((C.RUTA_RES / "recuperacion_parametros.json").read_text())
REC.alpha, REC.beta = PARAM["alpha"], PARAM.get("beta_sistema", 0.5)    # configuracion que usa el agente
RES_REC = pd.read_csv(RUTA_REC)
base = RES_REC.loc[RES_REC["metodo"] == "bm25", "nDCG@10 eval (candidatos anotados)"].iloc[0]
RES_REC["mejora vs BM25"] = RES_REC["nDCG@10 eval (candidatos anotados)"] - base
print("Parametros elegidos en dev:", PARAM)
RES_REC.round(4)

## 6. Modelos de lenguaje y voz (carga inicial y memoria)

| Etapa | Modelo | Por qué | VRAM aprox. |
|---|---|---|---|
| ASR | `openai/whisper-large-v3-turbo` (fp16) | Mejor WER en español que small/medium con 4 capas de decodificador: rápido | ~1,6 GB |
| LLM / agente | `Qwen/Qwen2.5-7B-Instruct` (nf4) | Llamado a herramientas nativo (`<tool_call>`), buen español, cabe en 16 GB cuantizado | ~5,5 GB |
| Embeddings | `intfloat/multilingual-e5-base` | Multilingüe, buen equilibrio calidad/latencia | ~1,1 GB |
| Reranker | `BAAI/bge-reranker-v2-m3` (fp16) | Cross-encoder multilingüe de alto rendimiento | ~1,1 GB |
| TTS | XTTS-v2 (coqui-tts) | Clonación de voz en español con 6–10 s de referencia, sin entrenar | ~2 GB |
| Semejanza | `microsoft/wavlm-base-plus-sv` | x-vectors para verificar el parecido con la voz de referencia | CPU |

Gestión de memoria: el LLM va en 4 bits; los latentes de la voz de referencia se calculan una sola vez; los tiempos y la
VRAM de cada carga quedan en la tabla `carga_modelos.csv`, separados de las consultas.

In [ ]:
from src.llm import LLMLocal
from src.voz import ASR, TTSClonado, Semejanza
MOD = {}
def obtener(nombre):
    """Carga perezosa: cada modelo se carga una vez y se registra su tiempo de carga inicial."""
    if nombre not in MOD:
        if nombre == "llm":
            MOD[nombre] = registrar_carga(C.MODELO_LLM, "agente", lambda: LLMLocal(C.MODELO_LLM, C.LLM_4BIT, C.LLM_MAX_TOKENS))
        elif nombre == "asr":
            MOD[nombre] = registrar_carga(C.MODELO_ASR, "asr", lambda: ASR(C.MODELO_ASR, DISP))
        elif nombre == "tts":
            assert C.RUTA_VOZ_REF.exists(), f"Falta la grabacion de referencia: {C.RUTA_VOZ_REF}"
            MOD[nombre] = registrar_carga(C.MODELO_TTS, "tts", lambda: TTSClonado(C.MODELO_TTS, C.RUTA_VOZ_REF, C.TEXTO_REF, DISP))
        elif nombre == "semejanza":
            MOD[nombre] = registrar_carga(C.MODELO_SEMEJANZA, "metricas", lambda: Semejanza(C.MODELO_SEMEJANZA, "cpu"))
    return MOD[nombre]

AUDIOS_FALTAN = []
from src.escenarios import ESCENARIOS
for e in ESCENARIOS:
    if e["modo"] == "voz":
        AUDIOS_FALTAN += [t["audio"] for t in e["turnos"] if not (C.RUTA_AUDIOS / t["audio"]).exists()]
VOZ_LISTA = C.RUTA_VOZ_REF.exists() and not AUDIOS_FALTAN
print("Voz de referencia:", "OK" if C.RUTA_VOZ_REF.exists() else "FALTA", "| audios de escenarios faltantes:", len(AUDIOS_FALTAN))
with BIT.paso("6. Carga de LLM"):
    obtener("llm")
if C.EJECUTAR_ESCENARIOS_VOZ and VOZ_LISTA:
    with BIT.paso("6.1 Carga de ASR y TTS"):
        obtener("asr"); obtener("tts")
pd.DataFrame(CARGA)

## 7. Agente, herramientas, memoria y validaciones
- **Herramientas:** `buscar_productos` (RAG), `ver_producto`, `consultar_atributo` (atributo no verificable = desconocido),
  `registrar_restricciones`, `agregar_al_carrito`, `cambiar_cantidad`, `ver_carrito`, `cotizar`, `confirmar_pedido`.
- **Memoria:** presupuesto, preferencias, carrito, cotización vigente, pedidos y productos mostrados; se inyecta en cada turno.
- **Validaciones por código:** enteros 1..`CANTIDAD_MAX`, stock, presupuesto, consentimiento explícito sobre una cotización ya
  mostrada, confirmación atómica (sin cambios parciales) e idempotente (sin pedidos duplicados).
- La autoprueba ejecuta todo esto sin modelos (BM25 + LLM con guion).

In [ ]:
import subprocess
with BIT.paso("7.1 Autoprueba sin modelos (python -m src.autoprueba)") as info:
    p = subprocess.run([sys.executable, "-m", "src.autoprueba"], cwd=C.RAIZ, capture_output=True, text=True)
    print(p.stdout[-800:], p.stderr[-1500:])
    assert p.returncode == 0, "fallo la autoprueba"

from src.escenarios import Sistema, ejecutar, factibilidad
SIS = Sistema(CAT, REC, TIENDA, MOD.get("llm"), C.K_RESULTADOS, C.MAX_PASOS_AGENTE, C.TURNOS_HISTORIAL)
with BIT.paso("7.2 Factibilidad de los 20 escenarios"):
    FAC = pd.DataFrame(factibilidad(SIS))
    FAC.to_csv(C.RUTA_RES / "escenarios_factibilidad.csv", index=False)
    assert FAC["factible"].all(), FAC[~FAC["factible"]]
FAC

In [ ]:
# Ejemplo de un turno de texto (estado restaurado al terminar)
SIS.reiniciar()
r = SIS.agente.turno("Busco unos auriculares bluetooth por menos de 20 euros")
print("Respuesta:", r["respuesta"])
for o in r["operaciones"]:
    print(" -", o["herramienta"], o["argumentos"], "->", "ok" if o["resultado"].get("ok") else o["resultado"].get("error"))
print({k: round(v, 3) for k, v in r.items() if k.startswith("t_")})
SIS.reiniciar()

## 8. Escenarios por texto (14)
Estado inicial reproducible (restaurar + preparación del escenario). Se guardan las operaciones y el estado final en
`output/escenarios/texto/<id>/`.

In [ ]:
CFG_ESC = {"datos": H_DATOS, "tienda": CFG_TIENDA, "rec": PARAM, "llm": C.MODELO_LLM, "4bit": C.LLM_4BIT,
           "pasos": C.MAX_PASOS_AGENTE, "hist": C.TURNOS_HISTORIAL, "k": C.K_RESULTADOS,
           "escenarios": hash_cfg(ESCENARIOS)}

def correr(modo, lista, carpeta, salida, **kw):
    def fn():
        res = []
        for e in lista:
            with BIT.paso(f"Escenario {e['id']} ({modo})", e["tipo"]) as info:
                try:
                    r = ejecutar(e, SIS, "voz" if modo == "voz" else "texto", C.RUTA_ESC / carpeta / e["id"], **kw)
                    info["comentario"] = "completado" if r["completado"] else f"fallo: {r['fallos'][:120]}"
                except Exception as ex:      # un escenario que falla no detiene los demas
                    r = {"id": e["id"], "modo": modo, "tipo": e["tipo"], "turnos": 0, "completado": False,
                         "fallos": f"excepcion: {type(ex).__name__}: {ex}", "incumplimientos": "", "pedidos": 0, "registro": []}
                    info["estado"], info["comentario"] = "ERROR", r["fallos"][:150]
                res.append(r)
        salida.write_text(json.dumps(res, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
    return fn

R_TEXTO = C.RUTA_RES / "escenarios_texto.json"
if C.EJECUTAR_ESCENARIOS_TEXTO:
    SIS.llm = obtener("llm")
    sint = C.TTS_EN_ESCENARIOS_TEXTO and C.RUTA_VOZ_REF.exists()
    BIT.ejecutar("8. Escenarios por texto", {**CFG_ESC, "tts": sint},  [R_TEXTO],
                 correr("texto", [e for e in ESCENARIOS if e["modo"] == "texto"], "texto", R_TEXTO,
                        tts=obtener("tts") if sint else None, sintetizar=sint))
RES_TEXTO = json.loads(R_TEXTO.read_text(encoding="utf-8")) if R_TEXTO.exists() else []
pd.DataFrame(RES_TEXTO)[["id", "tipo", "turnos", "completado", "pedidos", "fallos", "incumplimientos"]] if RES_TEXTO else "sin resultados"

## 9. Escenarios por voz (6) y repetición con la transcripción correcta
- Entrada: `input/audios/<id>_t<n>.wav` (ver `input/audios/GUION_GRABACION.md`). Salida por voz con XTTS y la voz de referencia.
- **E05:** el usuario revisa la transcripción y la corrige antes de ejecutar (se registra si la corrección era necesaria).
- **Repetición:** los mismos 6 casos con la transcripción correcta como entrada, para separar los errores del ASR de los del agente.

In [ ]:
VOZ = [e for e in ESCENARIOS if e["modo"] == "voz"]
R_VOZ, R_VOZ_TXT = C.RUTA_RES / "escenarios_voz.json", C.RUTA_RES / "escenarios_voz_transcripcion_correcta.json"
if C.EJECUTAR_ESCENARIOS_VOZ and not VOZ_LISTA:
    print("Faltan grabaciones; se omiten los escenarios por voz. Faltan:", ("referencia.wav " if not C.RUTA_VOZ_REF.exists() else "") + ", ".join(AUDIOS_FALTAN))
if C.EJECUTAR_ESCENARIOS_VOZ and VOZ_LISTA:
    SIS.llm = obtener("llm")
    h_aud = hash_cfg({a: (C.RUTA_AUDIOS / a).stat().st_size for e in VOZ for a in (t["audio"] for t in e["turnos"])})
    cfg_voz = {**CFG_ESC, "asr": C.MODELO_ASR, "tts": C.MODELO_TTS, "audios": h_aud, "ref": C.RUTA_VOZ_REF.stat().st_size}
    BIT.ejecutar("9.1 Escenarios por voz (audio)", cfg_voz, [R_VOZ],
                 correr("voz", VOZ, "voz", R_VOZ, ruta_audios=C.RUTA_AUDIOS, asr=obtener("asr"), tts=obtener("tts"), sintetizar=True))
    BIT.ejecutar("9.2 Escenarios por voz con transcripcion correcta", cfg_voz, [R_VOZ_TXT],
                 correr("texto", VOZ, "voz_transcripcion_correcta", R_VOZ_TXT))
RES_VOZ = json.loads(R_VOZ.read_text(encoding="utf-8")) if R_VOZ.exists() else []
RES_VOZ_TXT = json.loads(R_VOZ_TXT.read_text(encoding="utf-8")) if R_VOZ_TXT.exists() else []
if RES_VOZ:
    cmp = pd.DataFrame(RES_VOZ)[["id", "tipo", "turnos", "completado", "fallos"]].merge(
        pd.DataFrame(RES_VOZ_TXT)[["id", "completado", "fallos"]], on="id", suffixes=(" (audio)", " (transcripcion correcta)"))
    cmp["diferencia atribuible al ASR"] = cmp["completado (audio)"] != cmp["completado (transcripcion correcta)"]
    display(cmp)

## 10. Métricas
- **Escenarios:** % completados correctamente e incumplimientos de restricciones (presupuesto, cantidad, consentimiento,
  duplicado, stock negativo).
- **ASR:** WER/CER y errores en cantidades, importes y productos (referencia = transcripción correcta).
- **TTS:** inteligibilidad (WER de la re-transcripción de la respuesta sintetizada) y semejanza con la voz de referencia
  (coseno de x-vectors; referencia ≈ 0,86 para misma persona en WavLM-SV).
- **Latencia:** desde que termina la grabación hasta que la respuesta puede reproducirse, por etapa; la carga inicial de
  los modelos se reporta aparte.

In [ ]:
from collections import Counter
from src.voz import wer, cer, errores_entidades
TODOS = [dict(r, conjunto="texto") for r in RES_TEXTO] + [dict(r, conjunto="voz (audio)") for r in RES_VOZ] \
      + [dict(r, conjunto="voz (transcripcion correcta)") for r in RES_VOZ_TXT]
filas = []
for nombre, grupo in [("Oficial: 14 texto + 6 voz (audio)", RES_TEXTO + RES_VOZ), ("Texto", RES_TEXTO),
                      ("Voz (audio)", RES_VOZ), ("Voz (transcripcion correcta)", RES_VOZ_TXT)]:
    if grupo:
        inc = Counter(i for r in grupo for i in r["incumplimientos"].split(",") if i)
        filas.append({"conjunto": nombre, "escenarios": len(grupo), "completados": sum(r["completado"] for r in grupo),
                      "% completados": 100 * np.mean([r["completado"] for r in grupo]),
                      "incumplimientos": sum(inc.values()), "detalle incumplimientos": dict(inc)})
MET_ESC = pd.DataFrame(filas); MET_ESC.to_csv(C.RUTA_RES / "metricas_escenarios.csv", index=False)
display(MET_ESC.round(1))

# ASR por turno
ASR_T = pd.DataFrame([{"escenario": r["id"], "turno": t["turno"], "referencia": t["texto_correcto"],
                       "transcripcion": t["transcripcion_asr"], "wer": wer(t["texto_correcto"], t["transcripcion_asr"]),
                       "cer": cer(t["texto_correcto"], t["transcripcion_asr"]),
                       **errores_entidades(t["texto_correcto"], t["transcripcion_asr"]),
                       "correccion_necesaria": t.get("correccion_necesaria")}
                      for r in RES_VOZ for t in r["registro"]])
if len(ASR_T):
    ASR_T.to_csv(C.RUTA_RES / "asr_por_turno.csv", index=False)
    s = ASR_T.sum(numeric_only=True)
    MET_ASR = pd.DataFrame([{"turnos": len(ASR_T), "WER medio": ASR_T["wer"].mean(), "CER medio": ASR_T["cer"].mean(),
                             "errores cantidades": f"{int(s['err_cantidades'])} / {int(s['n_cantidades'])}",
                             "errores importes": f"{int(s['err_importes'])} / {int(s['n_importes'])}",
                             "errores productos (terminos)": f"{int(s['err_productos'])} / {int(s['n_productos'])}"}])
    MET_ASR.to_csv(C.RUTA_RES / "metricas_asr.csv", index=False); display(MET_ASR.round(3))

In [ ]:
# TTS: inteligibilidad y semejanza (se calcula una vez y se guarda)
R_TTS = C.RUTA_RES / "metricas_tts_por_respuesta.csv"
def medir_tts():
    asr, sem = obtener("asr"), obtener("semejanza")
    filas = []
    for r in RES_VOZ:
        for t in r["registro"]:
            wav = C.RUTA_ESC / "voz" / r["id"] / f"t{t['turno']}_asistente.wav"
            if wav.exists():
                hip = asr.transcribir(wav)
                filas.append({"escenario": r["id"], "turno": t["turno"], "texto": t["respuesta"], "re_transcripcion": hip,
                              "wer_inteligibilidad": wer(t["respuesta"], hip),
                              "semejanza_ref": sem.coseno(wav, C.RUTA_VOZ_REF)})
    ref2 = C.RUTA_VOZ_REF.with_name("referencia_2.wav")     # opcional: otra grabacion propia = techo de semejanza
    if ref2.exists():
        filas.append({"escenario": "techo", "turno": 0, "texto": "referencia_2 vs referencia", "re_transcripcion": "",
                      "wer_inteligibilidad": np.nan, "semejanza_ref": sem.coseno(ref2, C.RUTA_VOZ_REF)})
    pd.DataFrame(filas).to_csv(R_TTS, index=False)
if RES_VOZ:
    BIT.ejecutar("10.1 Metricas TTS (inteligibilidad y semejanza)", {"voz": hash_cfg(RES_VOZ), "sem": C.MODELO_SEMEJANZA}, [R_TTS], medir_tts)
    TTS_M = pd.read_csv(R_TTS)
    v = TTS_M[TTS_M["escenario"] != "techo"]
    display(pd.DataFrame([{"respuestas": len(v), "WER inteligibilidad medio": v["wer_inteligibilidad"].mean(),
                           "semejanza media": v["semejanza_ref"].mean(), "semejanza min": v["semejanza_ref"].min(),
                           "techo (otra grabacion propia)": TTS_M.loc[TTS_M["escenario"] == "techo", "semejanza_ref"].mean()}]).round(3))

In [ ]:
# Latencia por etapa (consultas) y carga inicial (separada)
ETAPAS = ["asr", "recuperacion", "llm", "herramientas", "tts"]
LAT = pd.DataFrame([{"conjunto": r["conjunto"], "escenario": r["id"], "turno": t["turno"], "pasos_llm": t["pasos_llm"],
                     **{k: t["tiempos"].get(k, np.nan) for k in ETAPAS + ["total"]}}
                    for r in TODOS for t in r["registro"]])
if len(LAT):
    LAT.to_csv(C.RUTA_RES / "latencias_por_turno.csv", index=False)
    resumen = LAT.groupby("conjunto")[ETAPAS + ["total", "pasos_llm"]].agg(["mean", "median", "max"]).round(2)
    resumen.to_csv(C.RUTA_RES / "latencias_resumen.csv"); display(resumen)
CARGA_DF = pd.DataFrame(CARGA); CARGA_DF.to_csv(C.RUTA_RES / "carga_modelos.csv", index=False)
print("Carga inicial de modelos (no incluida en la latencia por turno):"); display(CARGA_DF)

## 11. Gráficos

In [ ]:
with BIT.paso("11.1 Grafico: nDCG@10 por variante"):
    fig, ax = plt.subplots(figsize=(10, 4))
    cols = ["nDCG@10 dev", "nDCG@10 eval (candidatos anotados)", "nDCG@10 eval (catalogo completo)"]
    RES_REC.set_index("variante")[cols].plot.barh(ax=ax)
    ax.set_xlabel("nDCG@10"); ax.set_title("Recuperacion: modelo base, ablacion y sistema"); ax.invert_yaxis()
    guardar_fig(fig, "01_ndcg_variantes"); plt.show()
with BIT.paso("11.2 Grafico: ajuste en dev"):
    g = pd.read_csv(C.RUTA_RES / "recuperacion_ajuste_dev.csv")
    fig, ax = plt.subplots(figsize=(7, 4))
    for m, s in g.groupby("metodo"):
        x = s["alpha"] if m == "hibrido" else s["beta"]
        ax.plot(x, s["ndcg_dev"], marker="o", label=f"{m} ({'alpha' if m == 'hibrido' else 'beta'})")
    ax.set_xlabel("peso"); ax.set_ylabel("nDCG@10 dev"); ax.legend(); ax.set_title("Ajuste de pesos solo con dev")
    guardar_fig(fig, "02_ajuste_dev"); plt.show()

In [ ]:
if len(LAT):
    with BIT.paso("11.3 Grafico: latencia por etapa"):
        v = LAT[LAT["conjunto"] == "voz (audio)"] if (LAT["conjunto"] == "voz (audio)").any() else LAT
        fig, ax = plt.subplots(figsize=(11, 4))
        v.assign(t=v["escenario"] + "-t" + v["turno"].astype(str)).set_index("t")[ETAPAS].fillna(0).plot.bar(stacked=True, ax=ax)
        ax.set_ylabel("segundos"); ax.set_title("Tiempo desde el fin de la grabacion hasta la respuesta reproducible")
        guardar_fig(fig, "03_latencia_etapas"); plt.show()
if len(MET_ESC):
    with BIT.paso("11.4 Grafico: escenarios completados"):
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.barh(MET_ESC["conjunto"], MET_ESC["% completados"], color="tab:green"); ax.set_xlim(0, 100)
        ax.set_xlabel("% completados correctamente"); ax.invert_yaxis(); guardar_fig(fig, "04_escenarios"); plt.show()

## 12. Análisis de aciertos y fallos
Se extraen candidatos automáticamente (recuperación por consulta y escenarios). Completar la tabla con al menos
**5 aciertos y 5 fallos**, su causa probable y las limitaciones.

In [ ]:
pq = pd.read_csv(C.RUTA_RES / "recuperacion_por_consulta.csv")
sis_q = pq[pq["metodo"] == "sistema"].set_index("query_id")
bm_q = pq[pq["metodo"] == "bm25"].set_index("query_id")
dif = (sis_q["ndcg"] - bm_q["ndcg"]).rename("mejora_vs_bm25")
casos = pd.concat([sis_q[["query", "ndcg", "top3"]], dif], axis=1)
ACIERTOS_REC = casos.sort_values(["ndcg", "mejora_vs_bm25"], ascending=False).head(5).assign(tipo="Acierto recuperacion")
FALLOS_REC = casos.sort_values("ndcg").head(5).assign(tipo="Fallo recuperacion")
esc = pd.DataFrame([{"tipo": "Acierto escenario" if r["completado"] else "Fallo escenario", "query": f"{r['id']} ({r['conjunto']}) {r['tipo']}",
                     "ndcg": np.nan, "top3": r["fallos"] or "ok"} for r in TODOS])
CASOS = pd.concat([ACIERTOS_REC, FALLOS_REC, esc], ignore_index=False)
CASOS.to_csv(C.RUTA_RES / "aciertos_fallos.csv")
CASOS

### 12.1 Tabla de análisis (completar tras la ejecución)

| # | Tipo | Caso | Qué pasó | Causa probable |
|---|---|---|---|---|
| 1 | Acierto | | | |
| 2 | Acierto | | | |
| 3 | Acierto | | | |
| 4 | Acierto | | | |
| 5 | Acierto | | | |
| 6 | Fallo | | | |
| 7 | Fallo | | | |
| 8 | Fallo | | | |
| 9 | Fallo | | | |
| 10 | Fallo | | | |

### 12.2 Limitaciones
- Precios y existencias ficticios, independientes del tipo de producto (un precio puede no ser realista).
- El catálogo ESCI tiene 48 % de descripciones y 37 % de colores vacíos: muchos atributos quedan como desconocidos.
- La aclaración depende del criterio del LLM; las reglas duras (cantidad, stock, presupuesto, consentimiento) sí están en código.
- El consentimiento se detecta con patrones de afirmación/negación en español; frases inusuales pueden no reconocerse.
- XTTS-v2 tiene licencia no comercial (CPML); la semejanza depende de la calidad de la grabación de referencia.

## 13. Demo interactiva (micrófono, para el video)
Con `MODO_INTERACTIVO = True`: graba, muestra la transcripción, permite corregirla, ejecuta el agente, sintetiza la
respuesta con la voz clonada y la reproduce, mostrando la espera real por etapa. `SIS.reiniciar()` vuelve al estado inicial.

In [ ]:
from IPython.display import Audio, display
from src.voz import grabar
DEMO = C.RUTA_ESC / "demo"; DEMO.mkdir(parents=True, exist_ok=True)
LAT_DEMO = []

def turno_voz(segundos=C.SEGUNDOS_GRABACION, corregir=True):
    n = len(LAT_DEMO) + 1
    wav = grabar(DEMO / f"t{n}_usuario.wav", segundos, C.SR_ASR)        # el tiempo empieza al terminar la grabacion
    t0 = time.perf_counter(); texto = obtener("asr").transcribir(wav); t_asr = time.perf_counter() - t0
    print("Transcripcion:", texto)
    if corregir:
        c = input("Enter para aceptar o escribe la correccion: ").strip()
        texto = c or texto
    r = SIS.agente.turno(texto)
    t0 = time.perf_counter(); out = obtener("tts").sintetizar(r["respuesta"], DEMO / f"t{n}_asistente.wav", semilla=n); t_tts = time.perf_counter() - t0
    fila = {"turno": n, "asr": t_asr, "recuperacion": r["t_recuperacion"], "llm": r["t_llm"], "herramientas": r["t_herramientas"], "tts": t_tts}
    fila["total"] = sum(v for k, v in fila.items() if k != "turno"); LAT_DEMO.append(fila)
    print("Asistente:", r["respuesta"]); print({k: round(v, 2) for k, v in fila.items()})
    for o in r["operaciones"]:
        print(" -", o["herramienta"], o["argumentos"], "->", "ok" if o["resultado"].get("ok") else o["resultado"].get("error"))
    display(Audio(str(out), autoplay=True))

if C.MODO_INTERACTIVO:
    SIS.llm = obtener("llm"); SIS.reiniciar()
    print("Listo: ejecuta turno_voz() en una celda nueva por cada intervencion.")

## 14. Bitácora, instrucciones, versiones y configuración

In [ ]:
BIT_DF = pd.read_csv(BIT.csv)
ult = BIT_DF.groupby("paso", sort=False).tail(1)
lineas = ["# Tiempos medidos", "", "| ejecucion | paso | estado | tiempo usado |", "|---|---|---|---|"]
lineas += [f"| {r.ejecucion} | {r.paso} | {r.estado} | {r.tiempo_usado} |" for r in ult.itertuples()]
(C.RUTA_RES / "tiempos_medidos.md").write_text("\n".join(lineas) + "\n", encoding="utf-8")
print(json.dumps(VERSIONES, indent=1))
print({k: getattr(C, k) for k in dir(C) if k.isupper() and not k.startswith("RUTA") and k != "RAIZ"})
ult[["paso", "estado", "tiempo_usado", "comentario"]]

**Ejecución (local, equivalente a Colab)**
1. Entorno: `conda create -n ml4_env python=3.11 -y`, `conda activate ml4_env`, instalar torch con CUDA
   (`pip install torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124`) y luego
   `pip install -r requirements.txt`. En Colab: subir la carpeta a Drive, `%cd` a ella y `pip install -r requirements.txt`.
2. Grabar `input/voz/referencia.wav` (leer `TEXTO_REF`) y los 20 audios de `input/audios/` siguiendo `GUION_GRABACION.md`.
3. (Opcional) `python descargar_modelos.py` para dejar los modelos en caché antes de la primera ejecución.
4. Ejecutar el notebook de arriba a abajo. Lo ya hecho se omite; `FORZAR_REEJECUTAR = True` repite todo.
5. Demo del video: `MODO_INTERACTIVO = True` y llamar `turno_voz()`.

**Decisiones y diferencias con el enunciado**

| Requisito | Implementación |
|---|---|
| Procesamiento en Colab sin APIs externas | Notebook local equivalente; todos los modelos se descargan y corren en la máquina |
| ≥ 5 000 productos / ≥ 200 consultas | 10 688 productos, 250 eval (test) + 150 dev (train) con la configuración por defecto |
| Etiquetas no consultables | `qrels.parquet` en carpeta aparte; el agente solo ve catálogo y tienda |
| Validación por código | `src/tienda.py` (transacción, UNIQUE por cotización, CHECK stock ≥ 0) y `src/herramientas.py` |
| Modelo base y ablación | BM25 vs sistema; ablaciones sin reranker y sin denso, mismas consultas eval |
| Sin ajustar con evaluación | α y β se eligen en dev; eval se calcula una vez |